# 🧪 SDG Hub로 합성 데이터 생성하기

이 노트북에서는 합성 데이터 생성 파이프라인을 만들기 위한 모듈형 Python 프레임워크인 **SDG Hub**를 사용해 파인튜닝용 합성 학습 데이터를 생성하는 방법을 보여 줍니다.

> ⚠️ **워크숍 범위:** 전체 학습 데이터셋을 생성하려면 이번 워크숍에서 감당할 수 없는 시간과 API 비용이 듭니다. 이 섹션에서는 *과정*을 설명하고 도구를 보여 줍니다. 노트북에서는 작동 방식을 이해할 수 있도록 작은 샘플만 생성합니다. 실제 학습에는 부족하지만 파이프라인을 이해하기에는 충분합니다.

## 만들게 될 것

소크라테스식 수학 튜터를 위한 학습 데이터입니다. 각 예시는 다음과 같이 구성됩니다.
- 학생이 수학 질문을 합니다
- 튜터가 (답이 아니라) 생각을 이끄는 질문으로 답합니다

## 파이프라인

![SGD-pipeline.png](images/SGD-pipeline.png)

## SDG Hub 개념

| 개념 | 설명 |
|---------|-------------|
| **Blocks** | 데이터를 변환하는 개별 처리 단위 |
| **Flows** | 여러 블록을 연결하는 YAML 설정 기반 파이프라인 |
| **Registry** | 사용 가능한 flow와 block을 자동으로 찾아 주는 시스템 |

SDG Hub flow는 YAML로 정의되며 여러 단계를 조율합니다.

1. **문서 청크 분할** — 콘텐츠를 다루기 쉬운 크기로 나눕니다
2. **학생 질문 생성** — LLM이 콘텐츠를 바탕으로 실제 학생이 할 법한 질문을 만듭니다
3. **응답 생성** — LLM이 콘텐츠를 바탕으로 응답을 만듭니다
4. **대화 형식 구성** — 학습에 쓸 수 있도록 질문-답변 쌍을 구조화합니다

flow는 배포된 LLM 엔드포인트를 호출해 데이터를 생성합니다. 로컬 추론이 아니라 API 호출이므로 CPU 워크벤치에서도 문제없이 작동합니다.

---
## 0단계: Asyncio 설정

In [ ]:
# Jupyter 노트북에서 중첩 이벤트 루프 허용
import nest_asyncio
nest_asyncio.apply()

---
## 1단계: Docling으로 PDF를 마크다운으로 변환하기

소크라테스식 튜터링 예시가 담긴 Redwood Digital University의 실제 PDF 문서가 있습니다. Docling을 사용해 깔끔한 마크다운으로 변환해 보겠습니다.

In [ ]:
from docling.document_converter import DocumentConverter
import os

# 원본 PDF 경로
PDF_PATH = "Redwood_Digital_University_Socratic_Math_Tutor.pdf"

# PDF가 있는지 확인
if not os.path.exists(PDF_PATH):
    print(f"❌ PDF not found: {PDF_PATH}")
    print("Please ensure the PDF is in the same directory as this notebook.")
else:
    print(f"✅ Found PDF: {PDF_PATH}")
    print(f"   Size: {os.path.getsize(PDF_PATH) / 1024:.1f} KB")

In [ ]:
# Docling으로 PDF를 마크다운으로 변환
print("Converting PDF to markdown with Docling...")
print("(This may take a moment)\n")

converter = DocumentConverter()
result = converter.convert(PDF_PATH)

# 마크다운으로 내보내기
markdown_content = result.document.export_to_markdown()

print(f"✅ Conversion complete!")
print(f"   Markdown length: {len(markdown_content)} characters")
print(f"\n--- First 500 characters ---")
print(markdown_content[:500])

In [ ]:
# 참고용으로 마크다운 저장
markdown_path = "socratic-tutor-content.md"
with open(markdown_path, "w") as f:
    f.write(markdown_content)

print(f"✅ Saved markdown to: {markdown_path}")

---
## 2단계: 사용 가능한 SDG Hub Flow 찾기

SDG Hub는 레지스트리 시스템으로 사용 가능한 flow와 block을 자동으로 찾습니다. 어떤 것이 있는지 살펴보겠습니다.

In [ ]:
from sdg_hub import FlowRegistry, BlockRegistry, Flow

# 사용 가능한 flow 자동 탐색
FlowRegistry.discover_flows()

# 사용 가능한 모든 flow 목록 조회('id'와 'name' 키를 가진 dict 리스트 반환)
flows = FlowRegistry.list_flows()
print(f"Available flows ({len(flows)}):")
for flow_info in flows[:10]:  # 처음 10개만 표시
    print(f"  • {flow_info['id']}: {flow_info['name']}")
if len(flows) > 10:
    print(f"  ... and {len(flows) - 10} more")

In [ ]:
# 질문 생성 flow 검색('id'와 'name'을 가진 dict 리스트 반환)
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"Question generation flows ({len(qa_flows)}):")
for flow_info in qa_flows:
    print(f"  • {flow_info['id']}: {flow_info['name']}")

---
## 3단계: Flow 불러오고 설정하기

문서 기반 QA 생성 flow를 사용합니다. 이 flow는 문서 콘텐츠를 받아 질문-답변 쌍을 생성합니다.

In [ ]:
# QA 생성 flow 불러오기
# 프로덕션에서는 다음과 같이 사용합니다: flow = Flow.from_yaml(FlowRegistry.get_flow_path(flow_id))
# 이 데모에서는 내장 flow를 사용하거나 직접 설정합니다

try:
    # 문서 QA flow 불러오기 시도
    qa_flows = FlowRegistry.search_flows(tag="question-generation")
    if qa_flows:
        flow_id = qa_flows[0]['id']  # dict에서 flow ID 가져오기
        flow_path = FlowRegistry.get_flow_path(flow_id)
        flow = Flow.from_yaml(flow_path)
        print(f"✅ Loaded flow: {qa_flows[0]['name']} ({flow_id})")
    else:
        print("⚠️ No QA flows found, will use manual generation")
        flow = None
except Exception as e:
    print(f"⚠️ Could not load flow: {e}")
    print("   Will use manual generation instead")
    flow = None

In [ ]:
# flow가 있으면 모델 엔드포인트 설정
if flow:
    # 호스팅된 Llama 3.2 3B 엔드포인트를 사용하도록 설정
    # LiteLLM은 OpenAI 호환 엔드포인트에 "openai/" 접두사가 필요함
    MODEL_URL = "http://llama-32-predictor.ai501.svc.cluster.local:8080/v1"
    MODEL_NAME = "openai/llama32"  # LiteLLM provider 접두사 + 모델 이름
    API_KEY = "none"
    
    # flow의 모델 설정 지정
    flow.set_model_config(
        model=MODEL_NAME,
        api_base=MODEL_URL,
        api_key=API_KEY,
    )
    
    print(f"✅ Configured model:")
    print(f"   Provider: openai-compatible (LiteLLM)")
    print(f"   Model: {MODEL_NAME}")
    print(f"   Endpoint: {MODEL_URL}")

---
## 4단계: 데이터셋 준비하기

SDG Hub는 HuggingFace Datasets 형식을 사용합니다. flow에 넣을 문서 콘텐츠를 준비하겠습니다.

In [ ]:
from datasets import Dataset
import pandas as pd
import re

def chunk_by_sections(markdown: str) -> list[dict]:
    """헤더를 기준으로 마크다운을 섹션으로 나눕니다."""
    # 2단계 헤더(##)를 기준으로 분할
    sections = re.split(r'\n##\s+', markdown)
    
    chunks = []
    for i, section in enumerate(sections):
        if section.strip():
            # 첫 줄에서 제목 추출
            lines = section.strip().split('\n')
            title = lines[0].strip('#').strip() if lines else f"Section {i}"
            content = '\n'.join(lines[1:]).strip() if len(lines) > 1 else section.strip()
            
            if len(content) > 100:  # 내용이 충분한 섹션만 포함
                # 콘텐츠로 간단한 개요 생성
                outline_lines = [l.strip() for l in content.split('\n') if l.strip().startswith(('-', '*', '•'))]
                outline = '\n'.join(outline_lines[:5]) if outline_lines else f"Section about {title}"
                
                chunks.append({
                    'document': content,
                    'document_outline': outline,
                    'title': title,
                    'domain': 'Mathematics Education',
                    # 소크라테스식 튜터링을 위한 in-context learning 예시
                    'icl_document': 'A student learning to solve linear equations needs guidance, not answers.',
                    'icl_query_1': 'How do I solve 2x + 5 = 13?',
                    'icl_response_1': 'What operation would help you isolate the variable x?',
                    'icl_query_2': 'What is the derivative of x squared?',
                    'icl_response_2': 'What rule do you know for finding derivatives of power functions?',
                    'icl_query_3': 'I dont understand quadratic equations.',
                    'icl_response_3': 'What form does a quadratic equation typically take? Can you write one?',
                })
    
    return chunks

# 마크다운으로 청크 생성
chunks = chunk_by_sections(markdown_content)
print(f"✅ Created {len(chunks)} document chunks")

# 첫 번째 청크 표시
if chunks:
    print(f"\n--- First chunk ---")
    print(f"Title: {chunks[0]['title']}")
    print(f"Document preview: {chunks[0]['document'][:150]}...")
    print(f"Outline: {chunks[0]['document_outline'][:100]}...")

In [ ]:
# flow용 데이터셋 생성
if flow:
    # flow가 기대하는 스키마 가져오기(빈 pandas DataFrame 반환)
    schema_df = flow.get_dataset_schema()
    required_cols = list(schema_df.columns)
    print(f"Flow expects columns: {required_cols}")
    
    # 청크로 DataFrame 생성
    input_df = pd.DataFrame(chunks)
    
    # 데이터에 필요한 컬럼이 있는지 확인
    missing_cols = set(required_cols) - set(input_df.columns)
    if missing_cols:
        print(f"⚠️ Missing columns: {missing_cols}")
        print("   Adding placeholder values for missing columns...")
        for col in missing_cols:
            input_df[col] = f"[placeholder for {col}]"
    
    print("✅ All required columns present")
    input_dataset = input_df
else:
    # HuggingFace Dataset을 직접 생성(수동 생성용)
    input_dataset = Dataset.from_list(chunks)

print(f"\n✅ Dataset ready: {len(input_dataset)} examples")
cols = list(input_dataset.columns) if hasattr(input_dataset, 'columns') else input_dataset.column_names
print(f"   Columns: {cols}")

---
## 5단계: 합성 데이터 생성하기

이제 생성 파이프라인을 실행합니다. 프로덕션에서는 전체 SDG Hub flow를 사용합니다. 여기서는 두 가지 방식을 모두 보여 줍니다.

생성에 시간이 꽤 걸리므로, 여기서는 생성할 학습 예시 수를 2개로 제한합니다.

| 필요한 항목 | 워크숍 데모 | 프로덕션 |
|---------------|---------------|------------|
| 학습 예시 | 샘플 2개 | 500-5000개 |
| 생성 시간 | 약 2~5분 | 몇 시간에서 며칠 |
| API 호출 | 최소 | 수천 건 |

In [ ]:
# flow가 있으면 먼저 dry run 실행
if flow:
    print("Running dry run to estimate time...")
    dry_result = flow.dry_run(input_dataset, sample_size=1)
    print(f"\nDry run complete!")
    print(f"Estimated time for full generation: {dry_result.get('estimated_time', 'N/A')}")

In [ ]:
# 합성 Q&A 쌍 생성(데모용으로 샘플 수 제한)
SAMPLE_SIZE = 2  # 빠른 생성을 위해 샘플 2개로 제한

if flow:
    print(f"Generating synthetic data with SDG Hub (limited to {SAMPLE_SIZE} samples)...")
    print("(This may take a few minutes)\n")
    
    # 생성 속도를 높이기 위해 입력 데이터셋 크기 제한
    limited_dataset = input_dataset.head(SAMPLE_SIZE) if hasattr(input_dataset, 'head') else input_dataset[:SAMPLE_SIZE]
    
    result = flow.generate(limited_dataset)
    
    # 생성된 데이터를 추출해 학습 형식으로 변환
    questions = result['question'].tolist() if hasattr(result['question'], 'tolist') else list(result['question'])
    answers = result['response'].tolist() if hasattr(result['response'], 'tolist') else list(result['response'])
    
    print(f"✅ Generated {len(questions)} Q&A pairs")
    
    # 학습용 대화 형식으로 변환(수동 생성과 동일)
    training_data = []
    for q, a in zip(questions, answers):
        training_data.append({
            "messages": [
                {"role": "user", "content": q},
                {"role": "assistant", "content": a}
            ]
        })
    
    # 예시 표시
    for i in range(min(3, len(training_data))):
        print(f"\n--- Example {i+1} ---")
        print(f"Q: {training_data[i]['messages'][0]['content'][:100]}...")
        print(f"A: {training_data[i]['messages'][1]['content'][:100]}...")
else:
    print("⚠️ No SDG Hub flow available, will use manual generation in next cell")
    training_data = None  # 수동 생성으로 채워질 예정

---
## 6단계: 품질 필터링

생성된 데이터가 모두 좋은 것은 아닙니다. 응답이 실제로 소크라테스식 방법을 따르는지 확인하기 위해 필터링합니다.

프로덕션용 SDG Hub flow에서는 이러한 필터링이 품질 점수(faithfulness, relevancy)와 함께 파이프라인에 내장되어 있습니다.

In [ ]:
def is_socratic(response: str) -> bool:
    """응답이 소크라테스식 방법을 따르는지 확인합니다."""
    
    # 위험 신호 - 답을 직접 제공함(오탐을 피하기 위해 더 구체적으로 지정)
    answer_patterns = [
        "the answer is",
        "the solution is",
        "therefore, x =",
        "so x =",
    ]
    
    # 좋은 신호 - 생각을 이끄는 질문을 함
    question_patterns = [
        "what",
        "how",
        "why",
        "can you",
        "have you",
        "do you",
        "?"
    ]
    
    response_lower = response.lower()
    
    # 답을 제공하면 거부
    for pattern in answer_patterns:
        if pattern in response_lower:
            return False
    
    # 질문을 하면 수락
    for pattern in question_patterns:
        if pattern in response_lower:
            return True
    
    return False

# 먼저 필터링 전 응답이 어떤지 확인
print("Sample responses before filtering:")
for i, conv in enumerate(training_data[:5]):
    response = conv["messages"][1]["content"]
    is_ok = is_socratic(response)
    status = "✅" if is_ok else "❌"
    print(f"\n{status} Example {i+1}:")
    print(f"   Q: {conv['messages'][0]['content'][:80]}...")
    print(f"   A: {response[:150]}...")

# 필터 적용
filtered_data = [
    conv for conv in training_data
    if is_socratic(conv["messages"][1]["content"])
]

print(f"\n" + "="*60)
print(f"Quality filtering results:")
print(f"  Original: {len(training_data)} conversations")
print(f"  After filter: {len(filtered_data)} conversations")
print(f"  Kept: {len(filtered_data)/len(training_data)*100:.0f}%" if training_data else "N/A")

# 통과한 응답이 너무 적으면 경고하고 필터링을 건너뜀
if len(filtered_data) < 10 and len(training_data) > 10:
    print(f"\n⚠️  Very few responses passed the Socratic filter.")
    print(f"   The SDG Hub flow generates informative responses, not Socratic ones.")
    print(f"   For this demo, we'll use the unfiltered data.")
    filtered_data = training_data
    print(f"   Using all {len(filtered_data)} conversations.")

In [ ]:
# 몇 가지 예시로 필터 테스트
test_responses = [
    "The answer is x = 4",  # 실패해야 함
    "What operation would help you isolate x?",  # 통과해야 함
    "You get 8 when you subtract 5",  # 실패해야 함
    "What do you notice about the left side?"  # 통과해야 함
]

print("Quality filter test:")
for response in test_responses:
    result = "✅ PASS" if is_socratic(response) else "❌ FAIL"
    print(f"{result}: {response[:50]}...")

---
## 7단계: 학습 데이터 저장하기

학습 데이터는 JSONL 형식(한 줄에 JSON 객체 하나)으로 저장합니다. 학습 노트북에서 기대하는 형식입니다.

In [ ]:
import json

# JSONL로 저장
output_path = "socratic-training-data.jsonl"

with open(output_path, "w") as f:
    for conversation in filtered_data:
        f.write(json.dumps(conversation) + "\n")

print(f"✅ Saved {len(filtered_data)} examples to {output_path}")
print(f"   File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# 파일 내용 표시
print(f"\n--- File contents (first 3 lines) ---")
with open(output_path, "r") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line)
        print(f"\nExample {i+1}:")
        print(f"  Student: {data['messages'][0]['content'][:60]}...")
        print(f"  Tutor: {data['messages'][1]['content'][:60]}...")

---
## 요약

이 노트북에서는 작은 규모의 예제로 전체 합성 데이터 생성 파이프라인을 살펴보았습니다 📊  
실제 프로덕션 사례에서도 **과정**은 똑같으며, 규모만 달라집니다.

### 다음 단계

실습 안내 문서로 돌아가 이 데이터로 LoRA 학습을 하는 방법을 알아보세요!